# 04 — Facilities Bed Occupancy Silver

Builds `public_health.silver.facilities_bed_occupancy_silver` — anonymous hospital bed occupancy records.

### Source
- `public_health.bronze.facilities_bed_occupancy_bronze` (1,977 rows, 6 sheets: 2020–2025)

### Plan (facilities-silver-plan v1.5, section 3 + lineage contract v1.2)
- Grain: 1 row per anonymous hospital record (no facility matching)
- Key: `(_source_key, _source_file, _source_sheet, _source_row_num)` = lineage address
- Lineage: **7 columns** — `_silver_build_id` + address + `_as_of_date` + `_metric_source_columns`. Load time, load id, checksum and write time live in `quality.build_inputs` / `build_log` (reached via `v_build_lineage`), not on the rows
- Build sequence (contract 5.1): capture inputs → transform on captured version → validate candidate → write → append build records
- Every bronze row gets `row_disposition = 'loaded'` (eligibility is a flag, not a disposition)
- Write: Rebuild (CREATE OR REPLACE)
- All rows loaded including ineligible ones (eligibility is a flag, not a disposition)
- 2025 sheet: 1,162 rows (418 Gov + 744 Pvt), province level, with bed counts
- 2020–2024: 815 rows, Private only, no bed counts → `private_only_year` exclusion
- BOR thresholds: 0–100 normal, >100–150 WARN (eligible), >150 flagged (ineligible)
- `bor_bed_basis = 'authorized'`, `bor_basis_confirmed = false` (D6 pending)
- `reporting_period_start/end` = Jan 1 – Dec 31 of sheet year, `period_confirmed = false`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid
from datetime import datetime, timezone

BRONZE_TABLE = "public_health.bronze.facilities_bed_occupancy_bronze"
METADATA_TABLE = "public_health.bronze.ingestion_metadata"
TARGET_TABLE = "public_health.silver.facilities_bed_occupancy_silver"
BUILD_LOG_TABLE = "public_health.quality.build_log"
BUILD_INPUTS_TABLE = "public_health.quality.build_inputs"
ROW_DISPOSITION_TABLE = "public_health.quality.row_disposition"

SOURCE_KEY = "bed_occupancy"
EXPECTED_SHEETS = ["2020", "2021", "2022", "2023", "2024", "2025"]
EXPECTED_ROWS = 1977

SILVER_BUILD_ID = str(uuid.uuid4())
BUILD_STARTED_AT = datetime.now(timezone.utc)

print(f"Bronze table   : {BRONZE_TABLE}")
print(f"Target table   : {TARGET_TABLE}")
print(f"Silver build ID: {SILVER_BUILD_ID}")
print(f"Build started  : {BUILD_STARTED_AT.isoformat()}")

In [0]:
# v1.5 build sequence, step 1: capture inputs (plan 5.5, contract 5.1).
# One input per sheet, matched to exactly one processed ingestion_metadata row via
# (source_name, source_sheet, retrieved_at = _ingested_at) — contract 3.5.
bronze_history = spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}")
bronze_version = bronze_history.orderBy(F.desc("version")).first()["version"]
print(f"Bronze Delta version captured: {bronze_version}")

bronze_current = spark.table(BRONZE_TABLE)

# Distinct (sheet, ingested_at) pairs present in bronze, with row counts and file
sheet_loads = (
    bronze_current
    .groupBy("_source_sheet", "_ingested_at")
    .agg(
        F.first("_source_file", ignorenulls=True).alias("_source_file"),
        F.count("*").alias("input_rows"),
    )
)

meta = (
    spark.table(METADATA_TABLE)
    .filter((F.col("source_name") == SOURCE_KEY) & (F.col("status") == "processed"))
    .select(
        F.col("source_sheet").alias("meta_sheet"),
        F.col("retrieved_at").alias("meta_ingested_at"),
        F.col("batch_id"),
        F.col("content_checksum"),
        F.col("record_count"),
    )
)

captured_inputs = sheet_loads.join(
    meta,
    (F.col("_source_sheet") == F.col("meta_sheet"))
    & (F.col("_ingested_at") == F.col("meta_ingested_at")),
    how="left",
)

captured_rows = captured_inputs.orderBy("_source_sheet").collect()

print(f"\nCaptured inputs: {len(captured_rows)} sheets (expected {len(EXPECTED_SHEETS)})")
for r in captured_rows:
    print(f"  sheet {r['_source_sheet']}: rows={r['input_rows']}, batch={r['batch_id']}, "
          f"checksum={str(r['content_checksum'])[:16]}..., ingested_at={r['_ingested_at']}")

# Gates (plan 5.5 step 1): every declared sheet present, exactly one metadata match each
assert len(captured_rows) == len(EXPECTED_SHEETS), \
    f"Expected {len(EXPECTED_SHEETS)} captured inputs, got {len(captured_rows)}"
for r in captured_rows:
    assert r["_source_sheet"] in EXPECTED_SHEETS, f"Unexpected sheet: {r['_source_sheet']}"
    assert r["batch_id"] is not None, \
        f"Sheet {r['_source_sheet']}: no exactly-one processed ingestion_metadata match"
    assert r["content_checksum"] is not None, f"Sheet {r['_source_sheet']}: missing content_checksum"

input_rows_total = sum(r["input_rows"] for r in captured_rows)
print(f"\nInput rows across all sheets: {input_rows_total} (expected {EXPECTED_ROWS})")
assert input_rows_total == EXPECTED_ROWS, \
    f"Bronze rows {input_rows_total} != expected {EXPECTED_ROWS}"
print("Input capture gates passed")

In [0]:
# v1.5 build sequence, step 2: transform, reading the captured bronze version (VERSION AS OF).
bronze_asof = spark.sql(f"SELECT * FROM {BRONZE_TABLE} VERSION AS OF {bronze_version}")

# Cast BOR and bed counts to numeric
silver_df = (
    bronze_asof
    .withColumn("bor_pct", F.col("Bed_Occupancy_Rate_BOR_Based_on_Authorized_Beds").cast("double"))
    .withColumn("authorized_beds",
        F.when(F.col("Authorized_Bed_Capacity").isNull() | (F.col("Authorized_Bed_Capacity") == ""),
               F.lit(None))
        .otherwise(F.col("Authorized_Bed_Capacity").cast("int"))
    )
    .withColumn("implementing_beds",
        F.when(F.col("Implementing_Beds").isNull() | (F.col("Implementing_Beds") == ""),
               F.lit(None))
        .otherwise(F.col("Implementing_Beds").cast("int"))
    )
    .withColumn("reporting_year", F.col("_source_sheet").cast("int"))
)

total_rows = silver_df.count()
assert total_rows == input_rows_total, f"VERSION AS OF read {total_rows} rows, capture found {input_rows_total}"
print(f"Total bronze rows read (VERSION AS OF {bronze_version}): {total_rows}")
print(f"Rows with BOR: {silver_df.filter(F.col('bor_pct').isNotNull()).count()}")
print(f"Rows with authorized_beds: {silver_df.filter(F.col('authorized_beds').isNotNull()).count()}")
print(f"Rows with implementing_beds: {silver_df.filter(F.col('implementing_beds').isNotNull()).count()}")

In [0]:
# Identify exact duplicates (same business columns) - plan says 1 group in 2025 sheet only
# Earlier sheets: identical rows stay separate (they can be different hospitals, never de-duplicated)
from pyspark.sql.window import Window

dup_cols = ["Region", "Province/District", "City/Municipality", "General",
            "Nature_of_Ownership", "Bed_Occupancy_Rate_BOR_Based_on_Authorized_Beds",
            "Authorized_Bed_Capacity", "Implementing_Beds"]

# Only check duplicates within 2025 sheet (the only sheet with bed counts)
dup_window = Window.partitionBy(*dup_cols).orderBy("_source_row_num")

silver_df = (
    silver_df
    .withColumn("_dup_rank", F.row_number().over(dup_window))
    .withColumn("_dup_count", F.count("*").over(Window.partitionBy(*dup_cols)))
    .withColumn(
        "is_exact_duplicate",
        F.when(
            (F.col("reporting_year") == 2025)
            & (F.col("_dup_count") > 1)
            & (F.col("_dup_rank") > 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .drop("_dup_rank", "_dup_count")
)

dup_count = silver_df.filter(F.col("is_exact_duplicate")).count()
print(f"Exact duplicate rows flagged (2025 only): {dup_count}")

In [0]:
# Apply eligibility rules per plan section 3.2-3.3
# Ineligible reasons: bor_null, bor_negative, bor_gt_150, level_missing,
#   exact_duplicate, private_only_year (2020-2024)

silver_df = (
    silver_df
    .withColumn(
        "exclusion_reason",
        F.when(F.col("is_exact_duplicate"), F.lit("exact_duplicate"))
        .when(F.col("reporting_year") < 2025, F.lit("private_only_year"))
        .when(F.col("bor_pct").isNull(), F.lit("bor_null"))
        .when(F.col("bor_pct") < 0, F.lit("bor_negative"))
        .when(F.col("bor_pct") > 150, F.lit("bor_gt_150"))
        .when(F.col("General").isNull() | (F.col("General") == ""), F.lit("level_missing"))
        .otherwise(F.lit(None))
    )
    .withColumn(
        "baseline_eligible",
        F.when(F.col("exclusion_reason").isNull(), F.lit(True))
        .otherwise(F.lit(False))
    )
)

# Summary
print("Eligibility summary:")
silver_df.groupBy("baseline_eligible", "exclusion_reason") \
    .agg(F.count("*").alias("rows")) \
    .orderBy(F.desc("baseline_eligible"), "exclusion_reason") \
    .show(20, truncate=False)

eligible_2025 = silver_df.filter((F.col("baseline_eligible")) & (F.col("reporting_year") == 2025)).count()
print(f"\nBaseline-eligible 2025 rows: {eligible_2025}")

In [0]:
# Add BOR basis, reporting period, and confirmed flags (plan 3.2)
silver_df = (
    silver_df
    .withColumn("bor_bed_basis", F.lit("authorized"))
    .withColumn("bor_basis_confirmed", F.lit(False))
    .withColumn(
        "reporting_period_start",
        F.to_date(F.concat(F.col("reporting_year").cast("string"), F.lit("-01-01")))
    )
    .withColumn(
        "reporting_period_end",
        F.to_date(F.concat(F.col("reporting_year").cast("string"), F.lit("-12-31")))
    )
    .withColumn("period_confirmed", F.lit(False))
)

print("Reporting period sample:")
silver_df.select("reporting_year", "reporting_period_start", "reporting_period_end").distinct().orderBy("reporting_year").show()

In [0]:
# v1.5 build sequence, step 3: assemble and validate the candidate.
# Lineage (plan 5.2): 7 columns — silver build id, address, as-of date,
# and _metric_source_columns (every business column is renamed from bronze).
# No _load_batch_id / _content_checksum / _ingested_at / _silver_loaded_at on rows:
# they live in quality.build_inputs / build_log, reached via v_build_lineage.
metric_source_columns = F.create_map(
    F.lit("region"), F.lit("Region"),
    F.lit("province_district"), F.lit("Province/District"),
    F.lit("city_municipality"), F.lit("City/Municipality"),
    F.lit("hospital_level"), F.lit("General"),
    F.lit("ownership"), F.lit("Nature_of_Ownership"),
    F.lit("bor_pct"), F.lit("Bed_Occupancy_Rate_BOR_Based_on_Authorized_Beds"),
    F.lit("authorized_beds"), F.lit("Authorized_Bed_Capacity"),
    F.lit("implementing_beds"), F.lit("Implementing_Beds"),
)

candidate = silver_df.select(
    # Business columns
    F.col("reporting_year"),
    F.col("Region").alias("region"),
    F.col("Province/District").alias("province_district"),
    F.col("City/Municipality").alias("city_municipality"),
    F.col("General").alias("hospital_level"),
    F.col("Nature_of_Ownership").alias("ownership"),
    F.col("bor_pct"),
    F.col("authorized_beds"),
    F.col("implementing_beds"),
    F.col("bor_bed_basis"),
    F.col("bor_basis_confirmed"),
    F.col("reporting_period_start"),
    F.col("reporting_period_end"),
    F.col("period_confirmed"),
    F.col("baseline_eligible"),
    F.col("exclusion_reason"),
    F.col("is_exact_duplicate"),
    # Lineage: address + as-of date + source-column map + build id (plan 5.2)
    F.col("_source_key"),
    F.col("_source_file"),
    F.col("_source_sheet"),
    F.col("_source_row_num"),
    F.col("_as_of_date"),
    metric_source_columns.alias("_metric_source_columns"),
    F.lit(SILVER_BUILD_ID).alias("_silver_build_id"),
)

# Output validation on the candidate (plan 6.2): rows = bronze data rows, key unique/not null
key_cols = ["_source_key", "_source_file", "_source_sheet", "_source_row_num"]
cand_count = candidate.count()
dup_keys = candidate.groupBy(*key_cols).agg(F.count("*").alias("cnt")).filter(F.col("cnt") > 1).count()
null_keys = candidate.filter(
    F.col("_source_key").isNull() | F.col("_source_file").isNull()
    | F.col("_source_sheet").isNull() | F.col("_source_row_num").isNull()
).count()
lineage_cols = ["_silver_build_id", "_source_key", "_source_file", "_source_sheet",
                "_source_row_num", "_as_of_date", "_metric_source_columns"]
null_lineage = candidate.filter(" OR ".join(f"{c} IS NULL" for c in lineage_cols)).count()

print(f"Candidate rows: {cand_count} (expected {EXPECTED_ROWS})")
print(f"Duplicate keys: {dup_keys} (expected 0)")
print(f"Null keys: {null_keys} (expected 0)")
print(f"Rows missing any declared lineage column: {null_lineage} (expected 0)")

assert cand_count == EXPECTED_ROWS, f"Candidate has {cand_count} rows, expected {EXPECTED_ROWS}"
assert dup_keys == 0, f"{dup_keys} duplicate keys in candidate"
assert null_keys == 0, f"{null_keys} null keys in candidate"
assert null_lineage == 0, f"{null_lineage} rows missing declared lineage columns"
print("Candidate passed output validation (plan 6.2) — MERGE/write may proceed")

In [0]:
# v1.5 build sequence, step 4: write the candidate, then append the build records.
# build_log, build_inputs and row_disposition are appended only after the write succeeds.
candidate.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)
print(f"Table written: {TARGET_TABLE}")

target_version = spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE}").orderBy(F.desc("version")).first()["version"]
written_count = spark.table(TARGET_TABLE).count()
assert written_count == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS} rows in target, got {written_count}"

build_finished_at = datetime.now(timezone.utc)

# build_log: one row for this build (succeeded)
build_log_row = [(
    SILVER_BUILD_ID, "silver", TARGET_TABLE,
    BUILD_STARTED_AT, build_finished_at, "succeeded",
    int(written_count), int(target_version), None, None, None,
)]
spark.createDataFrame(build_log_row, schema=spark.table(BUILD_LOG_TABLE).schema) \
    .write.format("delta").mode("append").saveAsTable(BUILD_LOG_TABLE)

# build_inputs: one row per sheet read (plan 5.5 step 1: BOR has 6 sheets)
inputs_payload = [(
    SILVER_BUILD_ID, TARGET_TABLE, "bronze_source", BRONZE_TABLE, int(bronze_version),
    SOURCE_KEY, r["_source_file"], r["_source_sheet"], r["batch_id"], r["content_checksum"],
    r["_ingested_at"], int(r["input_rows"]),
) for r in captured_rows]
spark.createDataFrame(inputs_payload, schema=spark.table(BUILD_INPUTS_TABLE).schema) \
    .write.format("delta").mode("append").saveAsTable(BUILD_INPUTS_TABLE)

# row_disposition: every bronze data row is 'loaded' (plan 6.1 — eligibility is a flag,
# not a disposition; identical rows are never de-duplicated)
inputs_lookup = spark.createDataFrame(
    [(r["_source_sheet"], r["batch_id"], r["content_checksum"]) for r in captured_rows],
    ["sheet", "batch_id", "content_checksum"],
)
dispositions = (
    silver_df
    .join(inputs_lookup, F.col("_source_sheet") == F.col("sheet"), how="left")
    .select(
        F.lit(SILVER_BUILD_ID).alias("_silver_build_id"),
        F.col("_source_key"),
        F.col("_source_file"),
        F.col("_source_sheet"),
        F.col("_source_row_num"),
        F.col("batch_id").alias("_load_batch_id"),
        F.col("content_checksum").alias("_content_checksum"),
        F.lit(TARGET_TABLE).alias("target_table"),
        F.concat_ws("|", F.col("_source_file"), F.col("_source_sheet"),
                    F.col("_source_row_num").cast("string")).alias("target_key"),
        F.lit("loaded").alias("disposition"),
        F.lit(None).cast("string").alias("reason"),
    )
)
dispositions.write.format("delta").mode("append").saveAsTable(ROW_DISPOSITION_TABLE)

print(f"\nBuild records appended — build {SILVER_BUILD_ID}")
print(f"  build_log      : 1 row (succeeded, target Delta version {target_version})")
print(f"  build_inputs   : {len(inputs_payload)} sheet rows")
print(f"  row_disposition: {dispositions.count()} rows, all 'loaded'")

In [0]:
# Final validation on the published table (plan 6.2 output validation, 6.3 other checks)
result = spark.table(TARGET_TABLE)
total = result.count()
print(f"=== Validation: {TARGET_TABLE} ===")
print(f"Total rows: {total} (expected {EXPECTED_ROWS})")

# Key unique and not null
key_cols = ["_source_key", "_source_file", "_source_sheet", "_source_row_num"]
dup_keys = result.groupBy(*key_cols).agg(F.count("*").alias("cnt")).filter(F.col("cnt") > 1).count()
null_keys = result.filter(
    F.col("_source_key").isNull() | F.col("_source_file").isNull()
    | F.col("_source_sheet").isNull() | F.col("_source_row_num").isNull()
).count()
print(f"Duplicate keys: {dup_keys} (expected 0)")
print(f"Null keys: {null_keys} (expected 0)")

# Lineage completeness: the 7 declared columns (plan 5.4 always-group)
print("\nLineage completeness (7 declared columns):")
for c in ["_silver_build_id", "_source_key", "_source_file", "_source_sheet",
          "_source_row_num", "_as_of_date", "_metric_source_columns"]:
    null_ct = result.filter(F.col(c).isNull()).count()
    print(f"  {c}: {null_ct} nulls")
    assert null_ct == 0, f"{null_ct} nulls in {c}"

# Lineage join (plan 6.3): every silver row joins exactly one v_build_lineage row of its build
vbl = (
    spark.table("public_health.quality.v_build_lineage")
    .filter(F.col("build_id") == SILVER_BUILD_ID)
    .select(F.col("_source_key").alias("v_key"), F.col("_source_sheet").alias("v_sheet"))
)
missing_lineage = (
    result.select("_source_key", "_source_sheet").distinct()
    .join(vbl,
          (F.col("_source_key") == F.col("v_key")) & (F.col("_source_sheet") == F.col("v_sheet")),
          "left_anti")
    .count()
)
print(f"\nRows with no v_build_lineage match for this build: {missing_lineage} (expected 0)")

# Input accounting (plan 6.1): one disposition per bronze row read
build_disp = spark.table(ROW_DISPOSITION_TABLE).filter(F.col("_silver_build_id") == SILVER_BUILD_ID)
disp_count = build_disp.count()
print(f"row_disposition rows for this build: {disp_count} (expected {EXPECTED_ROWS})")

# Sheet breakdown
print("\nSheet breakdown:")
result.groupBy("_source_sheet", "ownership") \
    .agg(F.count("*").alias("rows"), F.sum(F.col("baseline_eligible").cast("int")).alias("eligible")) \
    .orderBy("_source_sheet", "ownership") \
    .show()

# BOR stats for 2025 eligible rows
print("BOR stats for 2025 eligible rows:")
result.filter((F.col("baseline_eligible")) & (F.col("reporting_year") == 2025)) \
    .select(F.col("bor_pct")).describe().show()

assert total == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS} rows, got {total}"
assert dup_keys == 0, f"{dup_keys} duplicate keys"
assert null_keys == 0, f"{null_keys} null keys"
assert missing_lineage == 0, f"{missing_lineage} rows with no v_build_lineage match"
assert disp_count == EXPECTED_ROWS, f"{disp_count} dispositions, expected {EXPECTED_ROWS}"
print("\n\u2705 All v1.5 validation checks passed")